In [2]:
import glob
from scipy.io import loadmat

import os
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, PathSolver

from sklearn.metrics import mean_squared_error, mean_absolute_error,root_mean_squared_error

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import math
import random

import sys

sys.path.append(os.path.abspath('./src'))
from scene_generation.core import Scene
from scene_generation.utils import rect_from_point_and_size, get_utm_epsg_code_from_gps, gps_to_utm_xy, get_center_subarray
from scene_generation.empirical_pathloss_model import pathloss_38901
from scene_generation.unet.unet_model_rt import UNet

# Mitsuba components for advanced scene operation
import mitsuba as mi
import drjit as dr

In [3]:
def scene_2d_height_data(mi_scene, size_box, resolution=200):
    """
    Compute a 2D top-down grid of building heights in the scene.
    
    Returns
    -------
    x, y : 1D numpy arrays
        Grid coordinates along X and Y
    height_grid : 2D numpy array
        Height values (NaN where no building)
    bbox_min, bbox_max : bounding box corners
    """
    # Calculate bounding box excluding ground
    bbox = mi.ScalarBoundingBox3f()
    for shape in mi_scene.shapes():
        if "ground" not in shape.id():
            bbox.expand(shape.bbox())

    bbox_min, bbox_max = bbox.min, bbox.max
    bbox_min.x = -size_box/2
    bbox_min.y = -size_box/2
    bbox_max.x = size_box/2
    bbox_max.y = size_box/2
    
    # Create grid
    #x = np.linspace(bbox_min.x, bbox_max.x, resolution)
    #y = np.linspace(bbox_min.y, bbox_max.y, resolution)
    x = np.linspace(-size_box/2, size_box/2, resolution)
    y = np.linspace(-size_box/2, size_box/2, resolution)
    xx, yy = np.meshgrid(x, y)

    # Rays start above scene
    z_start = bbox_max.z + 10
    origins = np.stack([xx.flatten(), yy.flatten(), 
                        np.full(xx.size, z_start)], axis=1)
    directions = np.tile([0, 0, -1], (len(origins), 1))

    # Intersect rays
    rays = mi.Ray3f(
        mi.Vector3f(np.array(origins, dtype=np.float32).T),
        mi.Vector3f(np.array(directions, dtype=np.float32).T)
    )
    si = mi_scene.ray_intersect(rays)
    
    heights = np.where(si.is_valid(), si.p.z, np.nan)
    height_grid = heights.reshape(xx.shape)

    return x, y, height_grid, bbox_min, bbox_max

In [4]:
import numpy as np
import sionna.rt
from sionna.rt import SceneObject, ITURadioMaterial
import mitsuba as mi

def check_position_valid(scene, position, car_size=2.0):
    """
    Check if a position is valid (not overlapping with buildings).
    
    Args:
        scene: Sionna RT scene
        position: (x, y, z) position to check
        car_size: approximate size of the car for collision checking
    
    Returns:
        bool: True if position is valid (no overlap), False otherwise
    """
    # Create a small offset to check multiple points around the car footprint
    check_radius = car_size * 0.5
    check_points = [
        position,  # center
        (position[0] + check_radius, position[1], position[2]),
        (position[0] - check_radius, position[1], position[2]),
        (position[0], position[1] + check_radius, position[2]),
        (position[0], position[1] - check_radius, position[2]),
    ]
    
    # Check if any point is inside a building using ray casting
    for point in check_points:
        # Cast a ray downward from above to check if point is inside geometry
        ray_origin = mi.Point3f(float(point[0]), float(point[1]), 50.0)
        ray_direction = mi.Vector3f(0.0, 0.0, -1.0)
        
        try:
            # Check intersection with scene geometry
            its = scene.mi_scene.ray_intersect(mi.Ray3f(ray_origin, ray_direction))
            
            # If intersection z is above ground level, position is inside/under a building
            if its.is_valid() and its.p.z > 0.5:
                return False
        except:
            # If ray test fails, be conservative and reject position
            return False
    
    return True

def add_random_cars_to_scene(scene, num_cars, area_radius, rx_locations):
    """
    Add random cars to the scene without overlapping with buildings.
    
    Args:
        scene: Sionna RT scene
        num_cars: number of cars to add
        area_radius: radius of the area to place cars
        rx_locations: array of valid RX positions (N x 3)
    
    Returns:
        tuple: (updated scene, list of car positions)
    """
    # Radio material constituting the cars
    car_material = ITURadioMaterial("car-material",
                                    "metal",
                                    thickness=0.01,
                                    color=(0.8, 0.1, 0.1))

    # Instantiate `num_cars` cars sharing the same mesh and material
    cars = [SceneObject(fname=sionna.rt.scene.low_poly_car,
                        name=f"car-{i}",
                        radio_material=car_material)
            for i in range(num_cars)]

    # Add the list of newly instantiated objects to the scene
    scene.edit(add=cars)

    min_dist_rx = 1.0   # Minimum distance from any RX
    max_dist_rx = 20.0  # Maximum distance from any RX
    car_size = 2.0      # Approximate car size for collision detection
    min_car_spacing = 3.0  # Minimum distance between cars

    # Generate a dense grid of candidate positions
    step = 2  # resolution of the grid (adjust as needed)
    xs = np.arange(-area_radius, area_radius+step, step)
    ys = np.arange(-area_radius, area_radius+step, step)
    grid_x, grid_y = np.meshgrid(xs, ys)
    candidates = np.column_stack([grid_x.ravel(), grid_y.ravel()])

    # Filter candidates based on constraints
    valid_xy = []
    for x, y in candidates:
        pos2d = np.array([x, y])
        pos3d = (float(x), float(y), 1.0)

        # Too close to TX?
        if np.linalg.norm(pos2d - np.array([0, 0])) < min_dist_rx:
            continue
        
        # Too close or too far from RX?
        diffs = np.linalg.norm(rx_locations[:, :2] - pos2d, axis=1)
        if np.any(diffs < min_dist_rx) or np.any(diffs > max_dist_rx):
            continue
        
        # Check for building overlap
        if not check_position_valid(scene, pos3d, car_size):
            continue
        
        valid_xy.append((x, y))

    valid_xy = np.array(valid_xy)

    if len(valid_xy) < num_cars:
        raise ValueError(f"Not enough valid positions available. Found {len(valid_xy)} valid positions but need {num_cars} cars.")

    # Randomly sample positions ensuring minimum spacing between cars
    car_positions = []
    available_indices = list(range(len(valid_xy)))
    
    for i in range(num_cars):
        if not available_indices:
            raise ValueError(f"Could not place all {num_cars} cars. Only placed {i} cars.")
        
        # Pick a random valid position
        idx = np.random.choice(available_indices)
        chosen_pos = valid_xy[idx]
        car_positions.append((chosen_pos[0], chosen_pos[1], 1.0))
        
        # Remove this position and nearby positions to maintain spacing
        remaining_indices = []
        for j in available_indices:
            dist = np.linalg.norm(valid_xy[j] - chosen_pos)
            if dist >= min_car_spacing:
                remaining_indices.append(j)
        available_indices = remaining_indices

    # Set positions/orientations
    for i in range(num_cars):
        cars[i].position = mi.Point3f(float(car_positions[i][0]),
                                      float(car_positions[i][1]),
                                      car_positions[i][2])
        cars[i].scaling = 2.0
        # Optional: add random rotation for variety
        cars[i].orientation = (0.0, 0.0, np.random.uniform(0, 360))

    return scene, car_positions

In [5]:
def get_rx_locations(mi_scene, num_rx, rx_height,size_box):
    """
    Generate exactly num_rx valid receiver positions in the scene.
    Returns the first num_rx valid positions found.
    
    Parameters:
    -----------
    scene : mitsuba scene object
    num_rx : int
        Number of receiver positions to generate
    rx_height : height of rx in (m)
    tx_height : height of tx in (m)
    
    Returns:
    --------
    selected_positions : numpy array of shape (num_rx, 3)
        First num_rx valid receiver positions
    """
    
    # Calculate scene bounding box excluding the ground plane
    bbox = mi.ScalarBoundingBox3f()
    for shape in mi_scene.shapes():
        if "ground" not in shape.id():
            bbox.expand(shape.bbox())

    bbox_min = bbox.min
    bbox_max = bbox.max
    low = [-size_box*0.5,-size_box*0.5]
    high = [size_box*0.5,size_box*0.5]
    
    # Generate random locations
    rand_x_y = np.random.uniform(low=low, high=high, size=(num_rx*10, 2))

    # Add constant rx height (z-coordinate)
    candidates = np.column_stack((rand_x_y, np.full((len(rand_x_y),), rx_height)))
    directions_np = np.tile(np.array([0, 0, 1]), (len(candidates), 1))

    # Ray tracing validation
    rays = mi.Ray3f(
        mi.Vector3f(np.array(candidates, dtype=np.float32).T),
        mi.Vector3f(np.array(directions_np, dtype=np.float32).T)
    )
    si = mi_scene.ray_intersect(rays)
    valid_mask = ~si.is_valid()

    # Filter valid positions
    valid_positions = candidates[valid_mask]
    
    # Return first num_rx positions
    if len(valid_positions) < num_rx:
        raise ValueError(f"Could not generate {num_rx} valid positions. "
                        f"Only found {len(valid_positions)} valid positions. "
                        f"Try reducing num_rx or increasing the multiplier.")
    
    return valid_positions[:num_rx]


In [6]:
def radio_scene(scene, frequency):
    # Configure simulation parameters
    scene.frequency = frequency
    scene.synthetic_array = True  # Optimize for array calculations

    for radio_material in scene.radio_materials.values():
        radio_material.scattering_coefficient  = 0.4 # (can adjust)

    # Perform ray tracing using PathSolver()
    solver = PathSolver()
    paths = solver(scene, 
                max_depth=2, # (can adjust)
                los=True,
                specular_reflection=True,
                diffuse_reflection=True, 
                refraction=True,
                samples_per_src=int(1e3) # (can adjust)
    )
    return paths

In [7]:
import random
import pandas as pd
import numpy as np
from sionna.rt import ITURadioMaterial, PathSolver
import json

def randomize_materials(scene, num_iterations=2, frequency=28.5e9):
    material_names = [
    "concrete", "brick", "wood", "glass", "ceiling_board",
    "chipboard", "plywood", "marble", "metal"
    ]

    # Step 1: Create one material object per type (reuse later)
    material_objects = {}
    for mat_name in material_names:
        material_objects[mat_name] = ITURadioMaterial(
            name=f"{mat_name}_mat",
            itu_type=mat_name,
            scattering_coefficient=0.4,
            thickness=0.1
        )

    random_material_dict = {}

    for it in range(num_iterations):
        # Step 2: Randomly assign materials to scene objects
        scene_assignment = {}
        for obj_name in scene.objects:
            mat_type = random.choice(material_names)
            mat_obj = material_objects[mat_type]
            scene.get(obj_name).radio_material = mat_obj
            scene_assignment[obj_name] = mat_type
        print("Starting ray tracing for iteration", it+1)
        # Step 3: Run ray-tracing solver
        paths = radio_scene(scene, frequency)
        print("Completed ray tracing for iteration", it+1)
        random_material_dict[f"Random_It_{it}"] = paths

    return random_material_dict


In [8]:
# Frequencies [Hz] at which to compute the channel response
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, subcarrier_frequencies
def extract_cfr(dict_paths,num_subcarriers, subcarrier_spacing):
    cfr_dict = {}
    frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)
    for name in dict_paths:
        cfr_dict[name] = dict_paths[name].cfr(frequencies=frequencies,
                   normalize=True,
                   normalize_delays=True,
                   out_type="numpy")


    return cfr_dict

 
def extract_cir(dict_paths):
    cir_dict = {}
    for name in dict_paths:
        cir_dict[name] = dict_paths[name].cir(normalize_delays=True,out_type="numpy")

    return cir_dict


In [9]:
import signal
import time

class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException

def scene_gen_with_timeout(scene_instance, polygon_points_gps,scene_dir,max_error , retries=3, timeout=30):
    for attempt in range(retries):
        try:
            # Register timeout
            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(timeout)

            # Call your function
            scene_instance(
                polygon_points_gps,
                scene_dir,
                None,
                osm_server_addr="https://overpass-api.de/api/interpreter",
                lidar_calibration=False,
                generate_building_map=True,
                ground_material_type="mat-itu_concrete",
                rooftop_material_type="mat-itu_metal",
                wall_material_type="mat-itu_brick",
                max_error=max_error
            )

            # Cancel alarm if successful
            signal.alarm(0)
            return True

        except TimeoutException:
            print(f"Attempt {attempt+1} timed out after {timeout}s, retrying...")

    raise RuntimeError(f"Function failed after {retries} retries")


In [ ]:
def generate_scene(cen_lon, cen_lat, scene_dir, max_error, size=50):
    """
    Given a set of coordinates, generate a scene
    """
    scene_instance = Scene()

    polygon_points_gps = rect_from_point_and_size(
        cen_lat,
        cen_lon,
        "center", 
        size,
        size
    )
    min_lon, min_lat = polygon_points_gps[0]
    max_lon, max_lat = polygon_points_gps[2]
    print(
        f"Check the bbox at http://bboxfinder.com/#{min_lat:.{4}f},{min_lon:.{4}f},{max_lat:.{4}f},{max_lon:.{4}f}"
    )
    scene_gen_with_timeout(scene_instance, polygon_points_gps,scene_dir,max_error , retries=3, timeout=30)
    # Merge_shapes = false so that we can adjust material properties later
    scene = load_scene("{}/scene.xml".format(scene_dir), merge_shapes = False)
    return scene

In [11]:
import pyproj
from pyproj import Transformer, CRS

def split_bounding_box(cen_lat, cen_lon, size, n_boxes):

    # Step 1: Get the GPS corners of the big rectangle
    polygon_points_gps = rect_from_point_and_size(
        cen_lat,
        cen_lon,
        "center", 
        size,
        size
    )
    if polygon_points_gps[0] == polygon_points_gps[-1]:
        polygon_points_gps = polygon_points_gps[:-1]
    # polygon_points_gps is [(lon, lat), ...]
    
    # Step 2: Convert the corners to UTM
    # Use center to determine UTM zone
    center_lat = cen_lat
    center_lon = cen_lon
    utm_epsg = get_utm_epsg_code_from_gps(center_lon, center_lat)
    big_rect_utm = [gps_to_utm_xy(lon, lat, utm_epsg)[:2] for lon, lat in polygon_points_gps]

    # Step 3: Compute min/max in UTM
    xs, ys = zip(*big_rect_utm)
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)

    # Step 4: Divide rectangle into sub-boxes in UTM
    rows = int(math.sqrt(n_boxes))
    cols = math.ceil(n_boxes / rows)
    x_step = (max_x - min_x) / cols
    y_step = (max_y - min_y) / rows

    # Step 5: Compute center points of sub-boxes and convert to GPS
    transformer_to_gps = Transformer.from_crs(utm_epsg, "EPSG:4326", always_xy=True)
    centers_gps = []

    for i in range(rows):
        for j in range(cols):
            if len(centers_gps) >= n_boxes:
                break
            cen_x = min_x + (j + 0.5) * x_step
            cen_y = min_y + (i + 0.5) * y_step
            cen_lon, cen_lat = transformer_to_gps.transform(cen_x, cen_y)
            centers_gps.append((cen_lat, cen_lon))

    return centers_gps

In [12]:
import multiprocessing
import time

def _generate_scene_worker(cen_lat, cen_lon, subdir_path, max_error, size, return_dict):
    """
    Worker function to run in a separate process.
    """
    scene_sample = generate_scene(cen_lat, cen_lon, subdir_path, max_error, size=size)
    return_dict['scene'] = scene_sample

def generate_scene_with_timeout(cen_lat, cen_lon, subdir_path, max_error, size=50, timeout=100):
    """
    Keep trying generate_scene until it completes.
    Each attempt is terminated if it exceeds 'timeout' seconds.
    """
    while True:
        manager = multiprocessing.Manager()
        return_dict = manager.dict()
        p = multiprocessing.Process(
            target=_generate_scene_worker, 
            args=(cen_lat, cen_lon, subdir_path, max_error, size, return_dict)
        )
        p.start()
        p.join(timeout)  # wait for 'timeout' seconds

        if p.is_alive():
            print(f"Timeout after {timeout}s, killing process and retrying...")
            p.terminate()
            p.join()
            time.sleep(1)
            continue

        # Check if scene was returned
        if 'scene' in return_dict:
            return return_dict['scene']

        # Otherwise, retry
        print("No scene returned, retrying...")
        time.sleep(1)


In [13]:
import multiprocessing
multiprocessing.set_start_method('fork', force=True)


In [14]:
# import os
# import matplotlib.pyplot as plt
# from osmnx._errors import InsufficientResponseError  # import the OSMnx exception

# # Parameters
# boxes = split_bounding_box(40.350274, -74.657488, 500, 25)
# scene_dir = "./scenes"
# max_error = 0.05
# size = 100
# resolution = 200  # example: number of pixels along each axis

# # Generate scenes and plot
# for idx, (cen_lat, cen_lon) in enumerate(boxes):
#     # Scene folder
#     subdir_name = f"scene_{idx+1}"
#     subdir_path = os.path.join(scene_dir, subdir_name)
#     os.makedirs(subdir_path, exist_ok=True)

#     # Generate scene with skip on no-buildings
#     try:
#         scene_sample = generate_scene(cen_lat, cen_lon, subdir_path, max_error, size=size)
#     except InsufficientResponseError:
#         print(f"Skipping scene {idx+1} at ({cen_lat:.4f}, {cen_lon:.4f}) — no buildings found.")
#         continue  # skip to the next box

#     # Get 2D height data
#     x, y, height_grid, bbox_min, bbox_max = scene_2d_height_data(scene_sample.mi_scene, resolution)

#     # Determine extent from actual coordinates
#     extent = [x.min(), x.max(), y.min(), y.max()]

#     # Plot
#     plt.figure(figsize=(6,6))
#     plt.imshow(height_grid, origin='lower', cmap='terrain', extent=extent, aspect='auto')
#     plt.colorbar(label='Height (m)')
#     plt.title(f"Scene {idx+1} at ({cen_lat:.4f}, {cen_lon:.4f})")
#     plt.xlabel("X (m)")
#     plt.ylabel("Y (m)")
#     plt.show()



In [16]:
def get_valid_positions(mi_scene, num_points):

    # Calculate scene bounding box excluding the ground plane
    bbox = mi.ScalarBoundingBox3f()
    for shape in mi_scene.shapes():
        if "ground" not in shape.id():
            bbox.expand(shape.bbox())

    bbox_min = bbox.min
    bbox_max = bbox.max

    low = [bbox_min.x, bbox_min.y]
    high = [bbox_max.x, bbox_max.y]
    
    # Generate random locations
    rand_x_y = np.random.uniform(low=low, high=high, size=(num_points, 2))

    # Add constant rx height (z-coordinate)
    candidates = np.column_stack((rand_x_y, np.full((len(rand_x_y),), 1)))
    directions_np = np.tile(np.array([0, 0, 1]), (len(candidates), 1))

    # Ray tracing validation
    rays = mi.Ray3f(
        mi.Vector3f(np.array(candidates, dtype=np.float32).T),
        mi.Vector3f(np.array(directions_np, dtype=np.float32).T)
    )
    si = mi_scene.ray_intersect(rays)
    valid_mask = ~si.is_valid()

    valid_positions = candidates[valid_mask]
    return valid_positions


In [17]:
def random_material_RT(scene, frequency=28.5e9):
    material_names = [
    "concrete", "brick", "wood", "glass", "ceiling_board",
    "chipboard", "plywood", "marble", "metal"
    ]

    # Step 1: Create one material object per type (reuse later)
    material_objects = {}
    for mat_name in material_names:
        material_objects[mat_name] = ITURadioMaterial(
            name=f"{mat_name}_mat",
            itu_type=mat_name,
            scattering_coefficient=0.4,
            thickness=0.1
        )

    # Step 2: Randomly assign materials to scene objects
    scene_assignment = {}
    for obj_name in scene.objects:
        mat_type = random.choice(material_names)
        mat_obj = material_objects[mat_type]
        scene.get(obj_name).radio_material = mat_obj
        scene_assignment[obj_name] = mat_type
    paths = radio_scene(scene, frequency)

    return paths


In [18]:
def data_csi(paths_dict, scene, label, num_tx_antennas, frequency,resolution,tx_height,valid_rx_positions,size_box):
    
    scene.tx_array = PlanarArray(
        num_rows=num_tx_antennas, # readjust dimensions if needed
        num_cols=num_tx_antennas,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="iso",        # isotropic pattern
        polarization="V"      # vertical polarization
    )

    # Receive array (single dipole element)
    scene.rx_array = PlanarArray(
        num_rows=1,
        num_cols=1,
        vertical_spacing=0.5,
        horizontal_spacing=0.5,
        pattern="dipole",
        polarization="V"
    )

    # Get raw scene
    x, y, height_data, bbox_min, bbox_max = scene_2d_height_data(scene.mi_scene,size_box,resolution)

    low = [bbox_min.x, bbox_min.y]
    high = [bbox_max.x, bbox_max.y]
    mid_point = (low[0]+high[0])/2
    print(low,high)
    tx_position = mi.Point3f(float(mid_point), float(high[1]), float(tx_height))

    # Add transmitter to the top center of the scene facing downwards
    print("Tx position:",tx_position)
    tx = Transmitter(
        name = "tx",
        position = tx_position,
        look_at = [0, 0, 0], 
    )
    scene.add(tx)

    # Add rx location to scene
    for index, location in enumerate(valid_rx_positions):
        rx = Receiver(
            name = f"rx_{index}",
            position = mi.Point3f(float(location[0]), float(location[1]), float(location[2])),
            orientation = [0, 0, 0]
        )
        scene.add(rx)

    if label == "Scatter":
        paths = radio_scene(scene, frequency)
        paths_dict[label] = paths
        material_aug_paths = random_material_RT(scene, frequency=frequency)
        paths_dict["Material"] = material_aug_paths
    else:
        paths = radio_scene(scene, frequency)
        paths_dict[label] = paths

    return paths_dict, height_data #RX position is it global?

In [19]:
import numpy as np
import os

def save_sample_data(folder_name, sample_id, cfr_dict, cir_dict, paths_dict, fig_matrices):
    os.makedirs(folder_name, exist_ok=True)
    file_refs = {}
    
    # save dictionaries of arrays
    file_refs["cfr_file"] = f"{folder_name}/cfr_{sample_id}.npz"
    np.savez_compressed(file_refs["cfr_file"], **cfr_dict)
    
    file_refs["cir_file"] = f"{folder_name}/cir_{sample_id}.npz"
    np.savez_compressed(file_refs["cir_file"], **cir_dict)
    
    file_refs["paths_file"] = f"{folder_name}/paths_{sample_id}.npz"
    np.savez_compressed(file_refs["paths_file"], **paths_dict)
    
    # save "figure data" as 2D matrices
    for map_name, map_array in fig_matrices.items():
        map_path = f"{folder_name}/{map_name}_{sample_id}.npz"
        np.savez_compressed(map_path, data=map_array)
        file_refs[f"{map_name}_matrix"] = map_path
    
    return file_refs


In [27]:
def calculate_coverage_from_height_grid(height_grid, size, resolution, ground_threshold=0.0):
    """
    Calculate building coverage from a 2D height grid.
    
    Args:
        height_grid: 2D numpy array of heights
        size: Physical size of the scene (meters)
        resolution: Number of pixels along one axis
        ground_threshold: Height threshold to distinguish buildings from ground (meters)
    
    Returns:
        coverage_percentage: Percentage of scene covered by buildings
        building_area_m2: Actual building footprint area in square meters
        total_area_m2: Total scene area in square meters
    """
    
    # Count pixels above ground threshold (buildings)
    building_pixels = np.sum(height_grid > ground_threshold)
    total_pixels = height_grid.size
    
    coverage = (building_pixels / total_pixels)
    
    # Calculate actual areas (each pixel represents a square area)
    # pixel_area_m2 = (size / resolution) ** 2  # area per pixel in square meters
    # building_area_m2 = building_pixels * pixel_area_m2
    # total_area_m2 = size * size  # total scene area
    
    return coverage

In [21]:
def validate_tx_location(mi_scene, bbox_min, bbox_max, tx_height, min_clearance_radius, num_samples=36):
    """
    Validate that a transmitter position is not inside a building and has 
    sufficient clearance from buildings within a specified radius.
    
    Parameters:
    -----------
    mi_scene : mitsuba scene object
        The scene to validate against
    tx_position : array-like of shape (3,)
        [x, y, z] coordinates of the transmitter
    min_clearance_radius : float
        Minimum radius (in meters) that should be clear of buildings
    num_samples : int
        Number of points to sample on the circle boundary (default 36 = every 10°)
    
    Returns:
    --------
    is_valid : bool
        True if the location is valid
    reason : str
        Explanation if invalid, empty string if valid
    """

    low = [bbox_min.x, bbox_min.y]
    high = [bbox_max.x, bbox_max.y]
    mid_point = (low[0]+high[0])/2
    
    tx_position = [mid_point, high[1], tx_height]
    
    # Check 1: Verify TX is not inside a building
    # Cast ray upward from TX position
    ray_up = mi.Ray3f(
        mi.Vector3f(tx_position),
        mi.Vector3f([0, 0, 1])
    )
    si_up = mi_scene.ray_intersect(ray_up)
    
    if si_up.is_valid():
        return False, "TX position is inside a building"
    
    # Check 2: Generate points on circle boundary at TX height
    angles = np.linspace(0, 2*np.pi, num_samples, endpoint=False)
    circle_x = tx_position[0] + min_clearance_radius * np.cos(angles)
    circle_y = tx_position[1] + min_clearance_radius * np.sin(angles)
    circle_z = np.full_like(circle_x, tx_position[2])
    
    circle_points = np.column_stack([circle_x, circle_y, circle_z])
    
    # Cast rays straight up from each circle point to check if inside building
    directions_up = np.tile(np.array([0, 0, 1]), (len(circle_points), 1))
    
    rays = mi.Ray3f(
        mi.Vector3f(circle_points.T.astype(np.float32)),
        mi.Vector3f(directions_up.T.astype(np.float32))
    )
    
    si = mi_scene.ray_intersect(rays)
    
    # If any point on the circle hits a building above it, it's inside a building
    valid_mask = si.is_valid()
    if np.any(valid_mask):
        num_invalid = np.sum(valid_mask)
        return False, f"{num_invalid}/{num_samples} points on clearance boundary are inside buildings"
    
    return True, ""

In [22]:
def remove_rx_from_scene(scene):
    rx_names = [rx for rx in scene.receivers]
    for rx_name in rx_names:
        scene.remove(rx_name)
    return scene

def add_unique_coords(final_valid_positions, df, num_rx):
    unique_rx_id = df['rx_id'].unique()
    for rx_id in unique_rx_id:
        rx_coord = df[df['rx_id'] == rx_id]['rx_coord'].values[0]
        # tolerance is within 10 cm
        if not any(np.allclose(rx_coord, existing, atol=1e-1) for existing in final_valid_positions) and len(final_valid_positions) < num_rx:
            final_valid_positions.append(rx_coord)

# Parse paths into a structured dataset containing detailed rays information
def create_ray_dataset(paths, valid_positions, threshold_dB):
    """Process raw ray data into pandas DataFrame"""
    dataset = {
        'rx_id': [],
        'a':[],
        'path gain (dB)':[],
        'rx_coord':[]
    }
    
    a = np.asarray(paths.a).squeeze()
    types = np.asarray(paths.interactions).squeeze()
    mask = np.asarray(paths.valid).squeeze()
    
    # Extract relevant parameters
    for idx, item in enumerate(mask):
        # Filter out the RX if it has no valid paths
        if np.sum(item==True):
            for sub_idx, sub_item in enumerate(item):
                if sub_item:
                    cur_ray_type = 0
                    num_depths = types.shape[0]
                    for depth_idx in range(num_depths-1, -1, -1):
                        cur_ray_type = types[depth_idx,idx,sub_idx]
                        # print(f"types[{depth_idx},{idx},{sub_idx}]",cur_ray_type)
                        if cur_ray_type != 0:
                            break

                    # Store the list instead of np.array to avoid the space seperator in .csv
                    z = np.complex64(a[0,idx,sub_idx] + 1j*a[1, idx,sub_idx]) 
                    dataset["a"].append(z)
                    dataset["rx_coord"].append(valid_positions[idx].tolist())
                    dataset["path gain (dB)"].append(20 * np.log10(np.abs(z)))
                    dataset["rx_id"].append(f"rx_{idx}")
    
    df = pd.DataFrame(dataset)
    df = df[df["path gain (dB)"] >= threshold_dB]
    return df



In [ ]:
def syn_data_generation(cen_lat, cen_lon, size_main_box, num_samples, size_box, num_rx, num_tx_ant, freq, rx_height, tx_height, min_tx_clearance_radius, 
                        resolution, coverage_threshold, scatter_density,scatter_aug_num,area_radius,mat_aug_num,building_loc_err,loc_aug_num,folder,num_subcarriers, subcarrier_spacing):
    dataset_records = []
    boxes = split_bounding_box(cen_lat, cen_lon, size_main_box, num_samples)
    for box_idx, box_coord in enumerate(boxes):
        folder_name = f"./scenes/{folder}/box_{box_idx:03d}"
        scene_sample_gt = generate_scene(box_coord[0], box_coord[1], folder_name, 0, size_box)
        # Check building coverage and validate tx location:
        x, y, height_grid, bbox_min, bbox_max = scene_2d_height_data(scene_sample.mi_scene,resolution)
        coverage = calculate_coverage_from_height_grid
        tx_validity = validate_tx_location(scene_sample_gt.mi_scene, bbox_min, bbox_max, tx_height, min_tx_clearance_radius)

        if coverage >= coverage_threshold:
            print(f"Skipping box {box_idx} due to low building coverage: {coverage*100:.2f}%")
            continue
        if not tx_validity[0]:
            print(f"Skipping box {box_idx} due to invalid TX location: {tx_validity[1]}")
            continue
        
        for loc_it in range(loc_aug_num):
            folder_name = f"./scenes/{folder}/box_{box_idx:03d}_loc_{loc_it:03d}"
            scene_sample = generate_scene(box_coord[0], box_coord[1], folder_name, building_loc_err, size_box)
            for scatter_it in range(scatter_aug_num):
                
                # Generate enough valid rx positions
                valid_rx_final = []
                while (len(valid_rx_final) < num_rx):
                    remove_rx_from_scene(scene_sample_gt)
                    valid_rx_positions = get_rx_locations(scene_sample_gt.mi_scene, num_rx, rx_height, size_box)
                    for idx, position in enumerate(valid_rx_positions):
                        rx = Receiver(
                            name = f"rx_{idx}",
                            position = mi.Point3f(float(position[0]), float(position[1]), float(position[2])),
                            orientation = [0, 0, 0]
                        )
                        scene_sample_gt.add(rx)
                    paths = radio_scene(scene_sample_gt, freq)
                    df = create_ray_dataset(paths, valid_rx_positions, -140)
                    add_unique_coords(valid_rx_final, df, num_rx)
                remove_rx_from_scene(scene_sample_gt)

                scene_sample_sc,car_positions = add_random_cars_to_scene(scene_sample_gt, scatter_density, area_radius, valid_rx_final)
                for mat_it in range(mat_aug_num):
                    paths_dict = {}
                    
                    paths_dict, gt_fig = data_csi(paths_dict, scene_sample_gt, "No Scatter", num_tx_ant, freq,resolution,tx_height,valid_rx_final)
                    paths_dict, sc_fig = data_csi(paths_dict, scene_sample_sc, "Scatter", num_tx_ant, freq,resolution,tx_height,valid_rx_final)
                    x, y, sample_fig, bbox_min, bbox_max = scene_2d_height_data(scene_sample.mi_scene,resolution)

                    cfr_dict = extract_cfr(paths_dict,num_subcarriers, subcarrier_spacing)
                    cir_dict = extract_cir(paths_dict)
                    

                    folder_name_res = f"./results_in_scene/{folder}/box_{box_idx:03d}_loc_{loc_it:03d}"
                    sample_id = scatter_it * mat_aug_num + mat_it
                    file_refs = save_sample_data(
                    folder_name_res, sample_id, cfr_dict, cir_dict, paths_dict,
                    {"gt": gt_fig, "sample": sample_fig, "sc": sc_fig})

                    record = {
                        "sample_id": sample_id,
                        "box_idx": box_idx,
                        "box_coords": box_coord,
                        "loc_it": loc_it,
                        "scatter_it": scatter_it,
                        "car_positions": car_positions,
                        "mat_it": mat_it,
                        "num_rx": num_rx,
                        "num_tx_ant": num_tx_ant,
                        "freq": freq,
                        "scatter_density": scatter_density,
                        "area_radius": area_radius,
                        "num_subcarriers": num_subcarriers,
                        "subcarrier_spacing": subcarrier_spacing,
                        "building_loc_err": building_loc_err,
                        "folder_name": folder_name,
                        "folder_name_res": folder_name_res,
                        "valid_rx_positions": valid_rx_final,
                        # references to stored files
                        **file_refs
                    }
                    dataset_records.append(record)

    dataset_df = pd.DataFrame(dataset_records)
    dataset_df.to_parquet(f"./meta_data_in_scenes/{folder}/dataset_metadata.parquet", index=False)
    
    return dataset_df


In [24]:
# Read cities from file
worldcities_df = pd.read_csv("worldcities.csv")
top_50_cities = worldcities_df.sort_values(by="population", ascending=False).head(50)
top_20_us_cities = worldcities_df[worldcities_df['country'] == 'United States'].sort_values(by="population", ascending=False).head(20)
top_20_us_cities[['city', 'lat', 'lng', 'population']]

,city,lat,lng,population
14,New York,40.6943,-73.9249,18832416.0
33,Los Angeles,34.1141,-118.4068,11885717.0
52,Chicago,41.8375,-87.6866,8489066.0
86,Miami,25.7840,-80.2101,6113982.0
88,Houston,29.7860,-95.3885,6046392.0
94,Dallas,32.7935,-96.7667,5843632.0
102,Philadelphia,40.0077,-75.1339,5696588.0
121,Atlanta,33.7628,-84.4220,5211164.0
123,Washington,38.9047,-77.0163,5146120.0
159,Boston,42.3188,-71.0852,4355184.0


In [ ]:
new_york_dir = "./scenes/new_york"
os.makedirs(new_york_dir, exist_ok=True)
new_york_long = -73.9249
new_york_lat = 40.6943
scene = generate_scene(new_york_lat, new_york_lat, new_york_dir, 0.05, size=1000)

Check the bbox at http://bboxfinder.com/#40.6899,-73.9309,40.6987,-73.9189


Parsing buildings: 100%|██████████| 2451/2451 [00:09<00:00, 256.50it/s]


2025-10-14 02:26:25 WARN wrk2 [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).